4. Diagram your bronze/silver/gold pipeline and label, for each layer, who the primary consumer is
(engineers, analysts, executives).

# Bronze / Silver / Gold Pipeline

## Pipeline Diagram

```text
                         RAW SALES CSV
                              |
                              v
                  +-----------------------+
                  |     BRONZE LAYER      |
                  | sales_bronze          |
                  |                       |
                  | Raw data +            |
                  | ingestion_timestamp   |
                  +-----------------------+
                              |
                              | Cleaning & Validation
                              v
                  +-----------------------+
                  |     SILVER LAYER      |
                  | sales_silver          |
                  |                       |
                  | - Fix data types      |
                  | - Remove duplicates   |
                  | - Remove invalid rows |
                  +-----------------------+
                              |
                              | Aggregation
                              v
                  +-----------------------+
                  |      GOLD LAYER       |
                  | product_gold          |
                  |                       |
                  | - Total quantity      |
                  | - Total revenue       |
                  +-----------------------+

                              |
                              v
                    Business Reporting
```


| Layer      | What happens here?                                                                             | Primary Consumer   |
| ---------- | ---------------------------------------------------------------------------------------------- | ------------------ |
| **Bronze** | Raw sales data is stored with an ingestion timestamp. No business transformations are applied. | **Data Engineers** |
| **Silver** | Data types are fixed, duplicate orders are removed, and invalid/null records are filtered out. | **Data Analysts**  |
| **Gold**   | Clean Silver data is aggregated by product to calculate total quantity and total revenue.      | **Executives**     |



5. Recreate one part of your silver transformation using Lakeflow Designer's visual, no-code interface
and compare the experience to writing it in code.

# Lakeflow Designer vs Code-Based Silver Transformation

## Transformation Recreated

I recreated part of my Silver transformation using **Lakeflow Designer's visual
no-code interface**.

The transformation I recreated was:

1. Remove duplicate records based on `order_id`.
2. Filter out records where `order_id` is NULL.
3. Filter out records where `quantity` is NULL or less than/equal to 0.

### Original PySpark Code

```python
silver_df = (
    df
    .dropDuplicates(["order_id"])
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("quantity") > 0)
)

6. Chain bronze → silver → gold as a Lakeflow Job with proper task dependencies, and configure it to
run on a schedule.

## Lakeflow Job Summary

I created a Lakeflow Job to automate the complete **Bronze → Silver → Gold** pipeline.

- **Bronze:** Ingests raw sales data.
- **Silver:** Cleans, validates, and removes duplicates.
- **Gold:** Creates product-level revenue and quantity summaries.

I configured proper task dependencies so **Silver runs after Bronze** and **Gold runs after Silver**. I also added a schedule so the pipeline runs automatically without manual execution.

**job yaml
```
resources:
  jobs:
    New_Job_2026_08_26_18_48_27:
      name: New Job 2026-08-26 18:48:27
      schedule:
        quartz_cron_expression: 0 0 12 * * ?
        timezone_id: Asia/Calcutta
        pause_status: UNPAUSED
      tasks:
        - task_key: Broze
          notebook_task:
            notebook_path: /Workspace/Users/rahulpatel@cyntexa.com/Databricks-Training-Practice-Assignment/Day-6
              Assig Medallion Architecture/bronze
            source: WORKSPACE
        - task_key: silver
          depends_on:
            - task_key: Broze
          notebook_task:
            notebook_path: /Workspace/Users/rahulpatel@cyntexa.com/Databricks-Training-Practice-Assignment/Day-6
              Assig Medallion Architecture/silver
            source: WORKSPACE
        - task_key: gold
          depends_on:
            - task_key: silver
          notebook_task:
            notebook_path: /Workspace/Users/rahulpatel@cyntexa.com/Databricks-Training-Practice-Assignment/Day-6
              Assig Medallion Architecture/gold
            source: WORKSPACE
      queue:
        enabled: true
      performance_target: PERFORMANCE_OPTIMIZED

```